# 📊 LCMSeg_turkm — External Evaluation на FLORES-200

**Цель:** оценить качество морфологической сегментации на независимом тексте
(предложения FLORES-200 не входили в обучающий корпус).

**Метрики:**
- **Boundary F1** — точность предсказания границ морфем
- **Token Accuracy** — доля правильно размеченных символов (BMES)
- **Word Accuracy** — доля слов с полностью верной сегментацией
- **Coverage** — доля слов которые модель смогла разбить (не вернула целиком)

**Важно:** это NOT extrinsic evaluation (NMT BLEU) — мы оцениваем
саму сегментацию, а не её влияние на downstream задачу.

## 📦 Ячейка E1 — Установка и импорты

In [ ]:
!pip install -q pytorch-crf
import os, re, json, torch, torch.nn as nn
from torchcrf import CRF
from collections import Counter, defaultdict
print('✅ Готово')


## 📁 Ячейка E2 — Drive + загрузка FEMSeg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_BASE  = '/content/drive/MyDrive/TURKM_MORPH'
CORPUS_NAME = 'turkmen_80000_segmented'
MODEL_DIR   = os.path.join(MODEL_BASE, f'models_{CORPUS_NAME}')
BEST_PT     = os.path.join(MODEL_DIR, f'femseg_turkm_{CORPUS_NAME}_BEST.pt')

# Туркменский алфавит и гармония
FRONT_VOWELS = set('eiöü'); BACK_VOWELS = set('ayou')
ALL_VOWELS = FRONT_VOWELS | BACK_VOWELS
HARMONY_FRONT=0; HARMONY_BACK=1; HARMONY_CONSONANT=2

def norm_turkm(s):
    return str(s).replace('\u0148','\u00f1').replace('\u0147','\u00d1').replace('\ufeff','').strip()
def char_harmony_class(ch):
    c=ch.lower()
    if c in FRONT_VOWELS: return HARMONY_FRONT
    if c in BACK_VOWELS:  return HARMONY_BACK
    return HARMONY_CONSONANT
def word_harmony_class(word):
    for ch in reversed(word.lower()):
        cls=char_harmony_class(ch)
        if cls!=HARMONY_CONSONANT: return cls
    return HARMONY_CONSONANT
def is_vowel(ch): return 1 if ch.lower() in ALL_VOWELS else 0

TAGS=['B','M','E','S']; TAG2ID={t:i for i,t in enumerate(TAGS)}
ID2TAG={i:t for t,i in TAG2ID.items()}

class TurkmFEMSegCRF(nn.Module):
    def __init__(self,vocab_size,tagset_size,emb_dim=128,harmony_dim=16,
                 vowel_dim=8,word_harm_dim=16,hidden_dim=256,dropout=0.3):
        super().__init__()
        self.char_emb=nn.Embedding(vocab_size,emb_dim,padding_idx=0)
        self.harmony_emb=nn.Embedding(3,harmony_dim)
        self.is_vowel_emb=nn.Embedding(2,vowel_dim)
        self.word_harm_emb=nn.Embedding(3,word_harm_dim)
        in_dim=emb_dim+harmony_dim+vowel_dim+word_harm_dim
        self.lstm=nn.LSTM(in_dim,hidden_dim,num_layers=2,bidirectional=True,
                          batch_first=True,dropout=dropout)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden_dim*2,tagset_size)
        self.crf=CRF(tagset_size,batch_first=True)
    def forward(self,inp,harm,vow,wh,tags=None,mask=None):
        x=torch.cat([self.char_emb(inp),self.harmony_emb(harm),
                      self.is_vowel_emb(vow),self.word_harm_emb(wh)],dim=-1)
        x,_=self.lstm(x); x=self.dropout(x); emis=self.fc(x)
        if tags is not None: return -self.crf(emis,tags,mask=mask,reduction='mean')
        return self.crf.decode(emis,mask=mask)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt=torch.load(BEST_PT,map_location=device)
char2id=ckpt['char2id']
femseg=TurkmFEMSegCRF(vocab_size=len(char2id),tagset_size=4).to(device)
femseg.load_state_dict(ckpt['state_dict']); femseg.eval()
print(f'✅ FEMSeg: epoch={ckpt["epoch"]}  morph_acc={ckpt["val_morph"]:.4f}  device={device}')


## 📖 Ячейка E3 — Загрузка FLORES-200 (туркменский)

In [ ]:
import urllib.request

os.makedirs('/tmp/flores', exist_ok=True)

# Скачиваем весь архив FLORES-200
archive = '/tmp/flores/flores200.tar.gz'
if not os.path.exists(archive):
    print('Скачиваем FLORES-200...')
    os.system('wget -q --trust-server-names https://tinyurl.com/flores200dataset '
              '-O /tmp/flores/flores200.tar.gz')
    os.system('tar -xzf /tmp/flores/flores200.tar.gz -C /tmp/flores/')
    print('✅ Распаковано')

# Находим файл туркменского языка
import glob
tuk_files = glob.glob('/tmp/flores/**/tuk_Latn.devtest', recursive=True)
if not tuk_files:
    # Пробуем альтернативное расположение
    tuk_files = glob.glob('/tmp/flores/**/*tuk*', recursive=True)
print('Найденные файлы:', tuk_files)

tuk_path = tuk_files[0] if tuk_files else None
if tuk_path:
    with open(tuk_path, encoding='utf-8') as f:
        flores_sentences = [l.strip() for l in f if l.strip()]
    print(f'✅ Загружено: {len(flores_sentences)} предложений')
    print(f'\nПримеры:')
    for s in flores_sentences[:3]: print(f'  {s}')
else:
    print('⚠️  Файл не найден — вводим вручную через HuggingFace')
    from datasets import load_dataset
    ds = load_dataset('Muennighoff/flores200', 'tuk_Latn', split='devtest')
    flores_sentences = [item['sentence'] for item in ds]
    print(f'✅ Загружено через HuggingFace: {len(flores_sentences)} предложений')


## ✂️ Ячейка E4 — Сегментация FLORES-200

In [ ]:
from tqdm import tqdm

NO_SPLIT_STEMS = {'türkmen','türkmenistan','döwlet','garaşsyz',
                  'mekdep','taýýar','ýaýran','aýran'}
VALID_SHORT    = {'da','de','dan','den','a','e','ny','ni','dy','di',
                  'lar','ler','yň','iň','sy','si','ly','li','na','ne'}
MERGE_RIGHT    = {'in','un','ün','yn'}

def check_no_split(word):
    w=word.lower()
    if w in NO_SPLIT_STEMS: return True,word,''
    for stem in sorted(NO_SPLIT_STEMS,key=len,reverse=True):
        if w.startswith(stem) and len(word)>len(stem):
            return True,word[:len(stem)],word[len(stem):]
    return False,word,''

def postprocess(morphs):
    if len(morphs)<=1: return morphs
    merged,i=[],0
    while i<len(morphs):
        m=morphs[i]
        if m.lower() in MERGE_RIGHT and i+1<len(morphs):
            merged.append(m+morphs[i+1]); i+=2
        else: merged.append(m); i+=1
    result=[merged[0]]
    for m in merged[1:]:
        if m in ('ñ','ň','Ñ','Ň'): result[-1]+=m
        elif len(m)==1 and m.lower() not in VALID_SHORT: result[-1]+=m
        else: result.append(m)
    return result

def raw_seg(word,model,c2id,dev):
    chars=list(word)
    if not chars: return [word]
    def t(lst,dt): return torch.tensor([lst],dtype=dt).to(dev)
    inp=t([c2id.get(ch,c2id['<unk>']) for ch in chars],torch.long)
    harm=t([char_harmony_class(ch) for ch in chars],torch.long)
    vow=t([is_vowel(ch) for ch in chars],torch.long)
    wh=t([word_harmony_class(word)]*len(chars),torch.long)
    mask=torch.ones(1,len(chars),dtype=torch.bool).to(dev)
    with torch.no_grad(): paths=model(inp,harm,vow,wh,tags=None,mask=mask)
    pred=[ID2TAG[tt] for tt in paths[0]]
    morphs,buf=[],[]
    for ch,tag in zip(chars,pred):
        buf.append(ch)
        if tag in('E','S'): morphs.append(''.join(buf)); buf=[]
    if buf: morphs.append(''.join(buf))
    return morphs

def seg_word(word,model,c2id,dev):
    w=norm_turkm(word)
    hit,stem,suf=check_no_split(w)
    if hit:
        if not suf: return [stem]
        return [stem]+postprocess(raw_seg(suf,model,c2id,dev))
    return postprocess(raw_seg(w,model,c2id,dev))

# Извлекаем все слова из FLORES-200
pat = re.compile(r'[a-zA-ZçñöşüýžÇÑÖŞÜÝŽ]+')
all_words = []
for sent in flores_sentences:
    tokens = pat.findall(norm_turkm(sent))
    all_words.extend(tokens)

# Уникальные слова (длина > 3 для интересных случаев)
unique_words = sorted(set(w.lower() for w in all_words if len(w) > 3))
print(f'Всего токенов:        {len(all_words):,}')
print(f'Уникальных слов (>3): {len(unique_words):,}')

# Сегментируем все уникальные слова
dev_str = str(device)
segmented = {}
for word in tqdm(unique_words, desc='Сегментация'):
    morphs = seg_word(word, femseg, char2id, dev_str)
    segmented[word] = morphs

print(f'\n✅ Сегментировано: {len(segmented)} слов')
print(f'\nПримеры:')
for w,m in list(segmented.items())[:10]:
    seg_str = '@@ '.join(m) if len(m)>1 else m[0]
    print(f'  {w:<25} → {seg_str}')


## 📊 Ячейка E5 — Автоматические метрики external evaluation

Поскольку у нас нет золотого стандарта сегментации для FLORES-200,
используем **косвенные метрики качества**:

1. **Coverage** — доля слов разбитых на 2+ морфемы
2. **Harmony Consistency** — % морфем согласованных по гармонии
3. **Avg morphemes per word** — средняя длина морфемной цепочки
4. **Single-char morpheme rate** — % одиночных символов (ошибки)
5. **OOV rate** — доля слов не из обучающего словаря

In [ ]:
# Загружаем char2id из обучения для проверки OOV
char2id_path = os.path.join(MODEL_BASE,
    f'char2id_turkmen_80000_segmented.json')
if os.path.exists(char2id_path):
    with open(char2id_path, encoding='utf-8') as f:
        train_chars = set(json.load(f).keys())
else:
    train_chars = set(char2id.keys())

# Считаем метрики
total = len(segmented)
multi_morph   = sum(1 for m in segmented.values() if len(m) > 1)
single_char   = sum(1 for m in segmented.values()
                    for part in m if len(part) == 1
                    and part.lower() not in VALID_SHORT)
total_morphs  = sum(len(m) for m in segmented.values())
avg_morphs    = total_morphs / max(1, total)

# Harmony consistency
harmony_ok = 0; harmony_total = 0
for word, morphs in segmented.items():
    if len(morphs) < 2: continue
    stem_h = word_harmony_class(morphs[0])
    if stem_h == HARMONY_CONSONANT: continue
    for affix in morphs[1:]:
        affix_h = word_harmony_class(affix)
        if affix_h == HARMONY_CONSONANT: continue
        harmony_total += 1
        if affix_h == stem_h: harmony_ok += 1

# OOV символы
oov_words = sum(1 for w in unique_words
                if any(ch not in train_chars for ch in w))

print('='*55)
print('  EXTERNAL EVALUATION — FLORES-200 (tuk_Latn)')
print('='*55)
print(f'  Уникальных слов проанализировано:  {total:,}')
print(f'  Покрытие (слов с сегментацией):    '
      f'{multi_morph/total*100:.1f}%  ({multi_morph:,}/{total:,})')
print(f'  Среднее морфем на слово:            {avg_morphs:.2f}')
print(f'  Согласованность гармонии:           '
      f'{harmony_ok/max(1,harmony_total)*100:.1f}%  '
      f'({harmony_ok}/{harmony_total})')
print(f'  Ошибочных одиночных символов:       {single_char}')
print(f'  OOV слов (новые символы):           {oov_words}')
print('='*55)

# Сохраняем результаты
ext_results = {
    'corpus': 'FLORES-200 tuk_Latn devtest',
    'total_unique_words': total,
    'coverage_pct': round(multi_morph/total*100, 1),
    'avg_morphemes': round(avg_morphs, 2),
    'harmony_consistency_pct': round(harmony_ok/max(1,harmony_total)*100, 1),
    'single_char_errors': single_char,
    'oov_words': oov_words,
}
json_path = os.path.join(MODEL_BASE, 'external_eval_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(ext_results, f, ensure_ascii=False, indent=2)
print(f'\n✅ Сохранено: {json_path}')


## 💾 Ячейка E6 — Сохранение списка слов для Human Evaluation

In [ ]:
import csv

# Выбираем 300 слов для ручной проверки
# Стратегия: 100 коротких (3-5 букв) + 100 средних (6-9) + 100 длинных (10+)
short_words = [w for w in unique_words if 3 < len(w) <= 5][:100]
mid_words   = [w for w in unique_words if 5 < len(w) <= 9][:100]
long_words  = [w for w in unique_words if len(w) >= 10][:100]
eval_words  = short_words + mid_words + long_words

print(f'Коротких слов (4-5):  {len(short_words)}')
print(f'Средних слов (6-9):   {len(mid_words)}')
print(f'Длинных слов (10+):   {len(long_words)}')
print(f'Итого для оценки:     {len(eval_words)}')

rows = []
for word in eval_words:
    morphs = segmented.get(word, [word])
    seg_str = '@@ '.join(morphs) if len(morphs) > 1 else morphs[0]
    rows.append({
        'word':       word,
        'femseg_seg': seg_str,
        'n_morphemes': len(morphs),
        'correct':    '',   # эксперт: TRUE / FALSE / PARTIAL
        'gold_seg':   '',   # эксперт: правильная сегментация
        'comment':    '',   # эксперт: комментарий
    })

csv_path = os.path.join(MODEL_BASE, 'human_eval_words.csv')
with open(csv_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f,
        fieldnames=['word','femseg_seg','n_morphemes','correct','gold_seg','comment'])
    writer.writeheader()
    writer.writerows(rows)

print(f'\n✅ CSV для эксперта: {csv_path}')
print(f'\nПервые 10 строк:')
print(f'  {"Слово":<20} {"FEMSeg сегментация":<35} {"Морфем"}')
print('  '+'-'*60)
for r in rows[:10]:
    print(f'  {r["word"]:<20} {r["femseg_seg"]:<35} {r["n_morphemes"]}')
print(f'\n  Файл готов для отправки эксперту-лингвисту.')
print(f'  Эксперт заполняет колонки: correct / gold_seg / comment')
